In [1]:
from langchain_core.documents import Document

In [2]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [3]:
from langchain_huggingface.embeddings import HuggingFaceEndpointEmbeddings
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("Hugging_face_api_token")
hf_embeddings_api = HuggingFaceEndpointEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2",  # lightweight
    # model="Qwen/Qwen3-Embedding-8B",
    # task="feature-extraction",# heavyweight but uses a lot of quota token
    huggingfacehub_api_token=api_key,
)

/Users/kristalshrestha/Documents/Code/LLM_Scratch/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from langchain_community.vectorstores import Chroma

In [5]:
#step 2 and 3: Initialize Embedding model and create chroma vector store in memory

vectorstore=Chroma.from_documents(
    documents=docs,
    embedding=hf_embeddings_api,
    collection_name="Kristal_MMR_collection",
    # persist_directory="Kristal_chroma_db"
    # if persist_directory not mentioned,vector store will be stored in RAM
)

In [10]:
#Enable MMR in the retriever
retriever=vectorstore.as_retriever(
    search_type="mmr", #<-- This enables MMR
    search_kwargs={
        "k":3, # k=top results
        "lambda_mult":0.5 #lambda_mult=relevance-diversity balance
    }
)

**Parameter:** `lambda_mult` (between 0 and 1).
    *   `lambda_mult = 1`: Behaves like standard similarity search (maximizes relevance, minimizes diversity).
    *   `lambda_mult = 0`: Maximizes diversity.

In [11]:
query="What is langchain?"
results=retriever.invoke(query)

In [12]:
results

[Document(metadata={}, page_content='LangChain supports Chroma, FAISS, Pinecone, and more.'),
 Document(metadata={}, page_content='LangChain is used to build LLM based applications.'),
 Document(metadata={}, page_content='Embeddings are vector representations of text.')]

In [13]:
for i,doc in enumerate(results):
    print(f"\n--Result{i+1}---")
    print(doc.page_content) 


--Result1---
LangChain supports Chroma, FAISS, Pinecone, and more.

--Result2---
LangChain is used to build LLM based applications.

--Result3---
Embeddings are vector representations of text.
